# sci_viz 测试与用法手册

本 notebook 依次演示：**环境检查 → 数据准备 → 16 种图各画一遍 → 并排与布局 → **主题（整篇配色统一）** → 单独配色 → 标签与公式 → 样式 → 保存 → 一键自检**。
放在仓库根目录直接运行（放在子目录也行，下面的第一个单元格会自动找到仓库根目录）。

**三条记住就够用的规则**
1. 所有函数：第 1 个参数是数据，第 2 个位置参数是 `ax`（`joint` 是 `fig`），其余都是关键字参数
2. 返回 `ax` 的函数是 *axes 级*，可以放进任意子图；返回 `fig` 的（`joint`、`pair`、`hexbin`、`ridgeline`）是 *figure 级*，自己占一整张图
3. 不传颜色时，各图的默认配色都来自当前**主题**（`pl.set_theme(...)`，默认 `ocean`），整篇论文配色自动匹配；个别图要特殊配色，用 `palette=`（离散）/ `cmap=`（连续）单独指定

## 0. 环境与导入

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import sci_viz as pl       # 导入时自动启用 pubstyle 样式（先 pip install -e . 安装过）
ps = pl.ps                  # 样式工具：panel_label / clean_axis / mixed_label / mixed_legend

In [ ]:
# 检查字体是否都能被 matplotlib 找到（False 的话中文会变方块 / 英文不是 Times）
from matplotlib import font_manager as fm
names = {f.name for f in fm.fontManager.ttflist}
print("Times New Roman:", "Times New Roman" in names, "| SimSun:", "SimSun" in names)

## 1. 准备演示数据

本 notebook 里所有图都用这里的数据。**换成你自己的数据**时，只需要保证格式对得上：

| 格式 | 例子 | 用在 |
|---|---|---|
| 两个一维数组 | `x, y` | `regression` `joint` `hexbin` |
| 分组 dict | `{"A": 数组, "B": 数组}`（也可以是每列一组的 DataFrame） | `box` `bar` |
| 分组 (x, y) dict | `{"A": (x, y), ...}` | `ellipse` |
| DataFrame | 行 = 样本 / 类别，列 = 变量 / 系列 | `pair` `corr_heatmap` `grouped_bar` `stacked_bar` ... |

In [ ]:
rng = np.random.default_rng(42)

# --- 两个一维数组 ---
x1 = rng.uniform(0, 10, 40);  y1 = 2.0 * x1 + 5 + rng.normal(0, 2, 40)      # 强相关
x2 = rng.uniform(0, 10, 40);  y2 = 0.5 * x2 + 10 + rng.normal(0, 5, 40)     # 弱相关
xj = rng.normal(50, 10, 200); yj = 0.8 * xj + rng.normal(0, 5, 200)
xh, yh = rng.multivariate_normal([3.0, 6.0], [[0.2, 0.1], [0.1, 0.5]], 300).T

# --- 分组数据：dict {组名: 一维数组} ---
groups = {"Control": rng.normal(100, 10, 60), "Group A": rng.normal(90, 20, 60),
          "Group B": rng.normal(110, 15, 60), "Group C": rng.normal(105, 5, 60)}
groups3 = {"Control": rng.normal(15, 3, 20), "Treat A": rng.normal(20, 4, 20), "Treat B": rng.normal(12, 2, 20)}

# --- 分组 (x, y)：椭圆图 ---
xa = rng.uniform(0, 10, 50); ya = 1.5 * xa + 2 + rng.normal(0, 2, 50)
xb = rng.uniform(0, 10, 50); yb = -1.2 * xb + 15 + rng.normal(0, 2, 50)
groups_xy = {"Group A (High)": (xa, ya), "Group B (Low)": (xb, yb)}

# --- DataFrame：散点矩阵（带分组列） ---
n = 150
df_pair = pd.DataFrame({"Feature_A": rng.normal(10, 2, n), "Feature_B": rng.normal(20, 5, n),
                        "Feature_C": rng.normal(30, 8, n), "Group": rng.choice(["Control", "Treatment"], n)})
df_pair["Feature_B"] += 0.6 * df_pair["Feature_A"]
df_pair["Feature_C"] -= 0.4 * df_pair["Feature_B"]

# --- DataFrame：相关性热力图（6 个有相关结构的变量） ---
f1 = np.linspace(0, 10, 100); f2 = f1[::-1]; f3 = rng.normal(5, 2, 100)
df_corr = pd.DataFrame({
    "Var_A": f1 + rng.normal(0, .5, 100), "Var_B": .9 * f1 + rng.normal(0, .3, 100),
    "Var_C": f2 + rng.normal(0, .4, 100), "Var_D": .85 * f2 + rng.normal(0, .2, 100),
    "Var_E": .5 * f1 + .5 * f3 + rng.normal(0, 1, 100), "Var_F": rng.normal(10, 2, 100)})

# --- DataFrame：Q 版热力图（行 = 基因，列 = 组织） ---
df_q = pd.DataFrame(rng.normal(size=(10, 4)), index=[f"Gene_{i}" for i in range(10)],
                    columns=["Liver", "Muscle", "Brain", "Kidney"])

# --- DataFrame：并列 / 堆积柱状图 / 面积图 ---
df_time = pd.DataFrame({"Time": ["Day 1", "Day 3", "Day 7"], "Control": [10, 15, 12], "Treatment": [12, 22, 30]})
df_stack = pd.DataFrame({"Sample": ["Site A", "Site B", "Site C"], "Type_1": [40, 30, 20],
                         "Type_2": [35, 40, 30], "Type_3": [25, 30, 25]})
df_area = pd.DataFrame({"Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
                        "Shade 5": [39, 5, 31, 31, 32, 10], "Shade 4": [8, 35, 14, 18, 28, 18],
                        "Shade 3": [15, 12, 9, 7, 20, 32], "Shade 2": [23, 30, 13, 27, 12, 25],
                        "Shade 1": [15, 18, 33, 17, 8, 15]})          # 每行总和 = 100

# --- 双 Y 轴：日期字符串 + 两条量纲不同的曲线 ---
t = np.arange(240)
dates = pd.date_range("2000-01-01", periods=240, freq="MS").strftime("%Y-%m")
y_left = 10 + 3 * np.sin(t / 30) + rng.normal(0, 0.3, 240)
y_right = 6000 + 2500 * np.sin(t / 45) + 8 * t

# --- 雷达图 / 山脊图 ---
criteria = ["Acc", "Prec", "Rec", "F1", "AUC", "MCC"]
radar_data = {"RF": [0.75, 0.82, 0.78, 0.80, 0.65, 0.72], "SVM": [0.65, 0.85, 0.60, 0.75, 0.55, 0.88],
              "XGB": [0.88, 0.70, 0.75, 0.72, 0.68, 0.75]}
df_ridge = pd.DataFrame({"Year": np.repeat(range(2010, 2021), 100),
                         "Value": rng.normal(size=1100).cumsum() + np.tile(np.linspace(0, 10, 11), 100)})
print("数据准备完成")

## 2. 逐个画 16 种图

每个单元格 = 一种图的最小调用 + 值得知道的常用变体。想看某个函数的全部参数：`help(pl.regression)`。

### 2.1 关系类

**`regression(x, y)`** 散点 + 回归线 + 置信带 + R²/p/N。常用参数：`ci=0.95`、`show_stats=False`、`legend=False`、`palette=[散点, 线, 置信带]`。

In [ ]:
ax = pl.regression(x1, y1, title="Group A: Strong Correlation")

In [ ]:
# 变体：不显示统计量和图例；90% 置信带；自定义 3 个颜色
ax = pl.regression(x2, y2, title="Group B", show_stats=False, legend=False, ci=0.90,
                   palette=["#2a9d8f", "#e76f51", "#e9c46a"])

**`joint(x, y)`** 主图散点回归 + 上方/右侧边缘直方图。figure 级，返回 `fig`。

In [ ]:
fig = pl.joint(xj, yj, xlabel="Control Group X", ylabel="Outcome Y")

**`pair(df, vars, hue)`** 多变量散点矩阵（对角线为核密度）。figure 级。`hue` 可省略；`title=None` 不要标题。

In [ ]:
fig = pl.pair(df_pair, vars=["Feature_A", "Feature_B", "Feature_C"], hue="Group")

In [ ]:
# 变体：不分组、自定义标题和颜色
fig = pl.pair(df_pair, vars=["Feature_A", "Feature_B"], palette=["#457B9D"], title="No hue")

**`ellipse({组名: (x, y)})`** 分组散点 + 置信椭圆。`n_std=2.0` 约对应 95%。

In [ ]:
ax = pl.ellipse(groups_xy, title="Scatter with Confidence Ellipses")

**`hexbin(x, y)`** 六边形分箱 + 边缘直方图 + 颜色条。figure 级。`cmap` 换分箱色表，`gridsize` 控制蜂窝大小。

In [ ]:
fig = pl.hexbin(xh, yh, xlabel="sepal width (cm)", ylabel="sepal length (cm)")

In [ ]:
fig = pl.hexbin(xh, yh, cmap="viridis", gridsize=15, title="cmap='viridis'")

**`dual_axis(x, y1, y2)`** 双 Y 轴折线。返回 `(ax左, ax右)`。字符串横轴（日期文本）自动稀疏标注并旋转 90°，数值横轴按实际值。

In [ ]:
ax1, ax2 = pl.dual_axis(dates, y_left, y_right, xlabel="Date",
                        ylabel1="Savings Rate", ylabel2="# Unemployed", title="Dual axis (string x)")

In [ ]:
# 变体：数值横轴，自定义左右轴颜色
ax1, ax2 = pl.dual_axis(t, y_left, y_right, xlabel="Month index", ylabel1="Left", ylabel2="Right",
                        palette=["#264653", "#e76f51"])

### 2.2 矩阵类

**`corr_heatmap(df)`** 相关系数热力图。`triangle='lower'/'upper'` 只画一半，`method='spearman'` 换相关类型，`cmap=` 换色表。

In [ ]:
ax = pl.corr_heatmap(df_corr, title="Correlation Matrix")

In [ ]:
# 变体：Spearman + 只画下三角 + 自定义发散色表
ax = pl.corr_heatmap(df_corr, method="spearman", triangle="lower", cmap=["#2a9d8f", "#ffffff", "#e76f51"],
                     title="Spearman, lower triangle")

**`q_heatmap(df)`** Q 版圆角热力图。`high_color=` 只换“高值”那一端的颜色，`label_size=` 调字号。

In [ ]:
ax = pl.q_heatmap(df_q)

In [ ]:
ax = pl.q_heatmap(df_q, high_color="#E63946", corner_radius=0.45)

### 2.3 比较类

**`box(data)`** 分组箱线图。`data` 可以是 dict、每列一组的 DataFrame、数组列表（用 `labels=[...]` 指定组名）。

In [ ]:
ax = pl.box(groups, title="Experiment Results", ylabel="Value (Units)")

In [ ]:
# 变体：数组列表 + labels；DataFrame 输入；不显示均值
ax = pl.box([groups["Control"], groups["Group A"]], labels=["对照组", "实验组"], show_mean=False, ylabel="Value")

**`bar(data)`** 均值柱状图：柱高 = 均值，误差棒 = SEM，叠加抖动散点。`show_points=False` 去掉散点，`seed=` 固定抖动。

In [ ]:
ax = pl.bar(groups3, ylabel="Measured Value (Units)", seed=0)

In [ ]:
ax = pl.bar(groups3, show_points=False, palette="sci6", ylabel="Mean ± SEM")

**`grouped_bar(df, category)`** 并列柱状图：`category` 是横轴类别列，其余列各为一个系列（也可用 `series=[...]` 指定）。

In [ ]:
ax = pl.grouped_bar(df_time, "Time", ylabel="Intensity (a.u.)")

**`radar(data, criteria)`** 环形雷达图。`data = {模型名: 各指标的值}`；`rmax` 设数据最大刻度，`ring_palette` 改外圈色块。

In [ ]:
ax = pl.radar(radar_data, criteria)

### 2.4 构成类

**`donut(values, labels)`** 甜甜圈图。`center_text` 写圆环中心文字（可含换行），`width` 调厚度。

In [ ]:
ax = pl.donut([40, 30, 20, 10], ["Control", "Treatment A", "Treatment B", "Placebo"], center_text="Cell\nViability")

**`stacked_bar(df, category)`** 堆积柱状图，`stack=[...]` 指定要堆叠的列（自下而上）。

In [ ]:
ax = pl.stacked_bar(df_stack, "Sample", ylabel="Relative Abundance (%)")

**`smooth_area(df)`** 平滑百分比堆叠面积图。第 1 列是类别，其余列是各层，每行总和应为 100。`transition_ratio` 越小越像阶梯。

In [ ]:
ax = pl.smooth_area(df_area)

In [ ]:
ax = pl.smooth_area(df_area, transition_ratio=0.1, legend=False, title="transition_ratio=0.1")

### 2.5 分布类

**`ridgeline(df, by, column)`** 山脊图（figure 级）。依赖 joypy，且 joypy 目前与 pandas ≥ 3 不兼容。

In [ ]:
try:
    fig = pl.ridgeline(df_ridge, by="Year", column="Value", xlabel="Value Axis", title="Ridgeline")
except (ImportError, RuntimeError) as e:
    print("跳过山脊图：", e)

## 3. 并排与布局

核心原理只有一条：**axes 级函数的第二个参数就是 `ax`**，所以任何 matplotlib 能创建 `ax` 的方式都能用。
figure 级函数（`joint`、`pair`、`hexbin`、`ridgeline`）自带整张画布，要排进别的图里只有 `joint` 能通过 `subfigures` 做到。

### 3.1 最简单：1 × 2 并排（同类型）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
pl.regression(x1, y1, axes[0], title="Group A")
pl.regression(x2, y2, axes[1], title="Group B")
fig.tight_layout()

### 3.2 2 × 2：不同类型的图放在一起，并加 (a)(b)(c)(d) 面板标号

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0))
pl.regression(x1, y1, axes[0, 0], title="regression", legend=False)
pl.bar(groups3, axes[0, 1], title="bar", seed=0)
pl.box(groups, axes[1, 0], title="box")
pl.grouped_bar(df_time, "Time", ax=axes[1, 1], title="grouped_bar")        # 注意：grouped_bar 的 ax 要用关键字
for a, lab in zip(axes.ravel(), "abcd"):
    ps.panel_label(a, f"({lab})")
fig.tight_layout()

### 3.3 共享 Y 轴：几张图用同一刻度，便于直接比较

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharey=True)
for a, shift in zip(axes, [0, 3, 6]):
    pl.bar({k: v + shift for k, v in groups3.items()}, a, title=f"shift = {shift}", seed=0)
fig.tight_layout()

### 3.4 GridSpec：大小不一的格子（左边一张大图，右边三张小图）

In [ ]:
fig = plt.figure(figsize=(7.2, 4.0))
gs = GridSpec(2, 3, figure=fig, width_ratios=[2, 1, 1])
a_big = fig.add_subplot(gs[:, 0])          # 左边整列
a_tr = fig.add_subplot(gs[0, 1:])          # 右上，横跨两格
a_bl = fig.add_subplot(gs[1, 1])
a_br = fig.add_subplot(gs[1, 2])

pl.regression(x1, y1, a_big, title="regression")
pl.bar(groups3, a_tr, title="bar", seed=0)
pl.box(groups, a_bl, labels=list("CABD"), title="box")          # 窄子图：用短组名避免挤在一起
pl.stacked_bar(df_stack, "Sample", ax=a_br, legend=False, title="stacked")
for a, lab in zip([a_big, a_tr, a_bl, a_br], "abcd"):
    ps.panel_label(a, f"({lab})", x=-0.14 if a in (a_big, a_tr) else -0.3)   # 窄子图的标号要再往左挪
fig.tight_layout()

### 3.5 subplot_mosaic：用“字符画”描述布局（最直观）

In [ ]:
fig, axd = plt.subplot_mosaic([["reg", "reg", "bar"],
                               ["box", "stk", "bar"]], figsize=(7.2, 4.4))
pl.regression(x1, y1, axd["reg"], legend=False, title="regression")
pl.bar(groups3, axd["bar"], title="bar", seed=0)
pl.box(groups, axd["box"], labels=list("CABD"), title="box")
pl.stacked_bar(df_stack, "Sample", ax=axd["stk"], legend=False, title="stacked")
fig.tight_layout()

### 3.6 subfigures：把 figure 级的 `joint` 排进去

`fig.subfigures()` 把一张图切成独立的“子画布”，`joint` 可以直接画在子画布上；子画布里还能再放普通子图。

In [ ]:
# 两个 joint 并排
fig = plt.figure(figsize=(7.2, 3.1))
s1, s2 = fig.subfigures(1, 2, wspace=0.1)
pl.joint(xj, yj, s1, xlabel="Control Group X", ylabel="Outcome Y")
pl.joint(xj[::-1], -0.4 * yj + 100, s2, xlabel="Treatment Group X", ylabel="Outcome Y")
fig.suptitle("Comparative Joint Distribution", fontsize=11, fontweight="bold")

In [ ]:
# 左边一个 joint，右边上下两个普通图
fig = plt.figure(figsize=(7.2, 3.6))
left, right = fig.subfigures(1, 2, width_ratios=[1.2, 1], wspace=0.08)
pl.joint(xj, yj, left, xlabel="X", ylabel="Y")
a_top, a_bot = right.subplots(2, 1)
pl.bar(groups3, a_top, title="bar", seed=0)
pl.box(groups, a_bot, title="box")

### 3.7 直角坐标 + 极坐标（雷达图）并排

In [ ]:
fig = plt.figure(figsize=(7.2, 3.4))
a1 = fig.add_subplot(1, 2, 1)
a2 = fig.add_subplot(1, 2, 2, projection="polar")           # 雷达图需要极坐标 ax
pl.box(groups, a1, title="box")
pl.radar(radar_data, criteria, a2, ncol=3)
fig.tight_layout()

### 3.8 同一张图里放饼图和柱状图

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.0, 3.2), gridspec_kw={"width_ratios": [1.3, 1]})
pl.donut([40, 30, 20, 10], ["Control", "Treatment A", "Treatment B", "Placebo"], a1, center_text="Cell\nViability")
pl.bar(groups3, a2, seed=0, title="bar")
fig.tight_layout()

**布局小抄**

| 想要 | 用 |
|---|---|
| 规则的 m × n | `plt.subplots(m, n)` |
| 共用坐标轴 | `plt.subplots(..., sharey=True)` |
| 大小不一 / 跨格 | `GridSpec` 或 `plt.subplot_mosaic` |
| 排进 `joint` | `fig.subfigures(...)` |
| 极坐标（雷达） | `fig.add_subplot(..., projection="polar")` |
| 面板标号 (a)(b) | `ps.panel_label(ax, "(a)")` |
| 自动防重叠 | `fig.tight_layout()`，或创建时 `layout="constrained"` |

`pair` / `hexbin` / `ridgeline` 自带整张画布，不能嵌入别的图，单独存成图片后在排版软件里拼即可。

## 4. 主题：整篇论文的配色统一

**主题 = 一组相互匹配的颜色**（主色序列 `cat`、强调色 `accent`、连续色表 `seq`、发散色表 `div`）。
所有图的默认配色都从当前主题派生：柱/箱/饼用主色序列，拟合线和双轴的另一侧用强调色，热力图用发散色表，hexbin 和山脊图用连续色表。
所以只要整篇论文用同一个主题，几十张图的配色就是匹配的；个别图想特殊，再单独指定。

**配色优先级（高 → 低）**：单次调用 `palette=` / `cmap=` → `palettes.use()` 全局覆盖 → 主题的 `overrides` → 主题默认

### 4.1 内置主题一览（`*` = 当前主题）

In [ ]:
print("可用主题:", pl.palettes.list_themes(), "| 当前:", pl.palettes.get_theme())
fig = pl.palettes.show_themes()

### 4.2 同一批图，换主题：整套图的配色一起变

In [ ]:
fig = plt.figure(figsize=(11.5, 8.4))
for sf, theme in zip(fig.subfigures(3, 1), ["ocean", "sunset", "nature"]):
    pl.set_theme(theme)
    axes = sf.subplots(1, 4)
    pl.bar(groups3, axes[0], seed=0)
    pl.regression(x1, y1, axes[1], title=None, show_stats=False, legend=False)
    pl.corr_heatmap(df_corr, axes[2], annot=False, title=None)
    pl.donut([40, 30, 20, 10], list("ABCD"), axes[3], center_text=None, legend_title=None)
    sf.suptitle(f'pl.set_theme("{theme}")', x=0.01, ha="left", fontweight="bold")
    sf.subplots_adjust(left=0.05, right=0.97, top=0.82, bottom=0.2, wspace=0.55)
pl.set_theme("ocean")

### 4.3 `set_theme` 同时让 matplotlib 原生图也跟主题（`plt.plot` 的颜色循环）

In [ ]:
pl.set_theme("forest")
fig, ax = plt.subplots(figsize=(4.2, 2.8))
for k in range(5):
    ax.plot(np.arange(10), np.arange(10) * (k + 1) * 0.3 + k, label=f"line {k}")   # 没指定颜色
ax.legend(frameon=False, ncol=2); ps.clean_axis(ax)
pl.set_theme("ocean")

### 4.4 自定义主题：只给主色序列就够，其余自动推导

In [ ]:
pl.palettes.register_theme("mine", cat=["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"])
pl.set_theme("mine")

fig, axes = plt.subplots(1, 4, figsize=(11.5, 2.9))
pl.bar(groups3, axes[0], seed=0, title="bar")
pl.regression(x1, y1, axes[1], title="regression", show_stats=False, legend=False)
pl.corr_heatmap(df_corr, axes[2], annot=False, title="corr_heatmap")
pl.donut([40, 30, 20, 10], list("ABCD"), axes[3], center_text=None, legend_title=None)
fig.tight_layout()

想精细控制就把 `accent`（强调色）、`seq`（3 个锚点，浅 → 深）、`div`（5 个锚点：负 → 中性 → 正）也写上：

```python
pl.palettes.register_theme("mine2", cat=[...], accent="#E76F51",
                           seq=["#EEF5F5", "#2a9d8f", "#10403B"],
                           div=["#264653", "#8FB8B5", "#F7F4EA", "#F4A261", "#E76F51"])
```

### 4.5 主题不变，只让个别图用特殊配色（`overrides`）

In [ ]:
pl.set_theme("mine", overrides={"bar": "vivid3", "corr_heatmap": "viridis"})   # key = 函数名

fig, axes = plt.subplots(1, 3, figsize=(8.6, 2.9))
pl.bar(groups3, axes[0], seed=0, title="bar（特殊配色）")
pl.box(groups, axes[1], title="box（仍跟主题）")
pl.corr_heatmap(df_corr, axes[2], annot=False, title="corr_heatmap（特殊色表）")
fig.tight_layout()

pl.set_theme("ocean")                         # 回到默认主题（overrides 同时清空）

### 4.6 回到原 notebook 的配色：`classic`

```python
pl.set_theme("classic")     # 各图恢复原来的配色（彼此不统一，仅用于复现旧图）
pl.set_theme("ocean")       # 回到默认主题
```

## 5. 配色

三种层次：**单次指定**（`palette=` / `cmap=`）→ **注册后按名字用** → **全局覆盖**。

### 5.1 单次指定：配色名 / 颜色列表

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.8, 2.8), sharey=True)
pl.bar(groups3, axes[0], title="default", seed=0)
pl.bar(groups3, axes[1], palette="sci6", title='palette="sci6"', seed=0)
pl.bar(groups3, axes[2], palette=["#E63946", "#457B9D", "#1D3557"], title="palette=[...]", seed=0)
fig.tight_layout()

### 5.2 连续色表：`cmap=` 可以是名称、颜色列表、matplotlib 色表名

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))
pl.corr_heatmap(df_corr, axes[0], title="default (blue_red)", annot=False)
pl.corr_heatmap(df_corr, axes[1], cmap="viridis", title='cmap="viridis"', annot=False)
pl.corr_heatmap(df_corr, axes[2], cmap=["#2a9d8f", "#ffffff", "#e76f51"], title="cmap=[3 colors]", annot=False)
fig.tight_layout()

### 5.3 注册自己的配色，以后按名字用

In [ ]:
pl.register_palette("mine", ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"])
pl.register_cmap("mine_div", ["#2a9d8f", "#ffffff", "#e76f51"])        # 连续 / 发散色表

fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 3.0))
pl.bar(groups3, a1, palette="mine", title='bar, palette="mine"', seed=0)
pl.corr_heatmap(df_corr, a2, cmap="mine_div", annot=False, title='cmap="mine_div"')
fig.tight_layout()

### 5.4 全局覆盖：所有图的离散配色 / 色表一次改完，`use()` 恢复

In [ ]:
pl.palettes.use("okabe_ito")          # 之后所有 palette=None 的图都用 okabe_ito
fig, axes = plt.subplots(1, 3, figsize=(7.8, 2.8))
pl.bar(groups3, axes[0], title="bar", seed=0)
pl.box(groups, axes[1], title="box")
pl.donut([40, 30, 20, 10], ["A", "B", "C", "D"], axes[2], center_text=None, legend_title=None)
fig.tight_layout()
pl.palettes.use()                     # 取消全局覆盖，回到当前主题

### 5.5 预览所有已注册的配色；写错配色名会直接报错

In [ ]:
fig = pl.palettes.show()

In [ ]:
for bad in ["sci66", ["#12345"]]:
    try:
        pl.bar(groups3, palette=bad)
    except ValueError as e:
        print("ValueError:", e)
plt.close("all")

## 6. 标签、公式与中文

- 公式用 `$...$`（LaTeX 语法），字符串前加 `r`
- `title / xlabel / ylabel` 里同时有**中文和公式**时会自动拆段渲染（不会出现方块）
- 图例里的中文 + 公式混排用 `ps.mixed_legend`

In [ ]:
ax = pl.regression(x1, y1, legend=False, show_stats=False,
                   xlabel=r"样本量 $n$ (Sample size)",
                   ylabel=r"均方误差 $\mathrm{MSE}(\hat{\theta})$",
                   title=r"正态分布 $N(\mu,\ \sigma^2)$")

In [ ]:
# 图例里的中文 + 公式：手动画线，再用 ps.mixed_legend
fig, ax = plt.subplots(figsize=(3.8, 3.0))
tt = np.linspace(0, 10, 200)
ax.plot(tt, np.sin(tt), color="#0072B2", label=r"模型 A，$\hat{\beta}_{OLS}$")
ax.plot(tt, np.cos(tt), color="#D55E00", label=r"模型 B，$\hat{\beta}_{Ridge}$")
ps.clean_axis(ax)
ps.mixed_label(ax, r"时间 $t$ (month)", where="x")
ps.mixed_label(ax, r"数值 $f(t)$", where="y", offset=0.14)
ps.mixed_legend(ax, loc="lower left")                 # 不传 handles / labels 时自动取 ax 上的线

## 7. 切换样式（字体、公式字体）

`pl.use_style(...)` 重新应用样式。常用：`cn_font="SimHei"`（换中文字体）、`math_fontset="stix"`（公式改成接近 Times 的字形）。

In [ ]:
pl.use_style(math_fontset="stix")
ax = pl.regression(x1, y1, title='math_fontset="stix"')
pl.use_style()                                        # 恢复默认（cm 公式字体）

## 8. 保存

`pl.save(对象, 路径)`：对象可以是 `Figure` / `Axes` / 子画布 / `dual_axis` 返回的 `(ax1, ax2)`。
路径**不带扩展名** → 同时存 png 和 pdf；**带扩展名** → 只存这一种。

In [ ]:
Path("figures").mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
pl.regression(x1, y1, axes[0], title="Group A")
pl.regression(x2, y2, axes[1], title="Group B")
fig.tight_layout()

print(pl.save(fig, "figures/demo_regression"))                       # png + pdf
print(pl.save(fig, "figures/demo_regression_600.png", dpi=600))      # 只存 png
print(pl.save(pl.bar(groups3, seed=0), "figures/demo_bar", formats=("png", "svg")))

## 9. 一键自检

把 `examples/gallery.py` 里登记的每个演示都跑一遍，汇总成功 / 失败。以后新增了图并登记到 `GALLERY` 后，重跑这一格就是回归测试。

In [ ]:
# gallery.py 在仓库的 examples/ 下：从当前目录往上找
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "gallery.py").is_file()), None)
assert root is not None, "找不到 examples/gallery.py：请把 test.ipynb 放在仓库里再运行"
sys.path.insert(0, str(root / "examples"))
import gallery

rows = []
for name, fn in gallery.GALLERY.items():
    try:
        fig = fn()
        rows.append((name, "ok", ""))
    except Exception as e:
        rows.append((name, "FAIL", f"{type(e).__name__}: {e}"))
    finally:
        plt.close("all")

report = pd.DataFrame(rows, columns=["plot", "status", "error"])
try:
    from IPython.display import display
    display(report)
except ImportError:
    print(report)
print(f"\n{(report.status == 'ok').sum()} / {len(report)} 通过")